In [6]:
# Few-Shot Classification using DenseNet
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import os

# DenseNet Feature Extractor
def get_feature_extractor():
    """
    Returns a DenseNet-121 feature extractor that outputs a flat embedding vector.
    """
  
    densenet161 = models.densenet161(weights=models.DenseNet161_Weights.DEFAULT)
       
    backbone = densenet161.features  # main CNN part
    
    backbone.eval()
    
    class FeatureExtractor(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone

        def forward(self, x):
            with torch.no_grad():
                feats = self.backbone(x)
                feats = F.adaptive_avg_pool2d(feats, (1, 1))  # global avg pool
                feats = torch.flatten(feats, 1)               # flatten (B, 1024)
                feats = F.normalize(feats, p=2, dim=1)        # normalize
            return feats

    return FeatureExtractor(backbone)




In [7]:
# Image Preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# Feature Extraction Helper

def extract_features(model, image_tensor):
    """Extracts normalized DenseNet features from a single image tensor."""
    with torch.no_grad():
        feats = model(image_tensor.unsqueeze(0))  # add batch dim
    return feats


In [8]:
# Class Prototypes
def build_prototypes(model, support_set):
    """
    support_set: dict {class_name: list of image_tensors}
    returns: dict {class_name: prototype_tensor}
    """
    prototypes = {}
    for cls, imgs in support_set.items():
        feats = torch.stack([extract_features(model, img) for img in imgs])
        proto = F.normalize(feats.mean(dim=0, keepdim=True), p=2, dim=1)
        prototypes[cls] = proto
    return prototypes

# Classify Query Samples

def classify_queries(model, prototypes, query_set):
    """
    query_set: dict {query_id: image_tensor}
    returns: dict {query_id: predicted_class}
    """
    predictions = {}
    for qid, img in query_set.items():
        # Extract and flatten query feature
        q_feat = extract_features(model, img).view(-1)  # force 1D (1024,)
        
        sims = {}
        for cls, proto in prototypes.items():
            proto = proto.view(-1)  # also force 1D (1024,)
            sim = torch.dot(q_feat, proto).item()  # cosine similarity (since normalized)
            sims[cls] = sim
        
        # Pick most similar class
        pred_cls = max(sims, key=sims.get)
        predictions[qid] = pred_cls
    
    return predictions

In [11]:
# Example Usage 

if __name__ == "__main__":
    # Load model
    model = get_feature_extractor()

    # Paths
    input_root = "D:/MSCS_Research/FewShot _Evaluations/Inputs"
    support_root = f"{input_root}/8Way_5Shot"
    query_root = f"{input_root}/Query8"

    # Helper: load all images from folder
    def load_images_from_folder(folder_path):
        imgs = []
        for fname in os.listdir(folder_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                img = Image.open(os.path.join(folder_path, fname)).convert("RGB")
                imgs.append(transform(img))
        return imgs

    # Build support set: {class_name: [tensors]}
    support_set = {
        cls: load_images_from_folder(os.path.join(support_root, cls))
        for cls in os.listdir(support_root)
        if os.path.isdir(os.path.join(support_root, cls))
    }

    # Build query set: {filename: tensor}
    query_set = {
        fname: transform(Image.open(os.path.join(query_root, fname)).convert("RGB"))
        for fname in os.listdir(query_root)
        if fname.lower().endswith((".jpg", ".jpeg", ".png"))
    }

    # Compute prototypes
    print("Computing class prototypes...")
    prototypes = build_prototypes(model, support_set)

    # Predict query samples
    print("Classifying query samples...")
    predictions = classify_queries(model, prototypes, query_set)

    # Print results
    print("\nPredictions:")
    for qid, cls in predictions.items():
        print(f"  {qid}: {cls}")


Computing class prototypes...
Classifying query samples...

Predictions:
  Buruli Ulcers10.jpg: Mycetoma
  Buruli Ulcers11.jpg: Mycetoma
  Buruli Ulcers12.jpg: Mycetoma
  Buruli Ulcers13.jpeg: Buruli Ulcers
  Buruli Ulcers14.jpg: Buruli Ulcers
  Buruli Ulcers16.jpg: Buruli Ulcers
  Buruli Ulcers17.png: Buruli Ulcers
  Buruli Ulcers18.jpg: Buruli Ulcers
  Buruli Ulcers19.jpg: Buruli Ulcers
  Buruli Ulcers20.jpg: Leprosy
  Buruli Ulcers5.jpg: Buruli Ulcers
  Buruli Ulcers6.jpg: Buruli Ulcers
  Buruli Ulcers7.jpg: Buruli Ulcers
  Leprosy1.jpg: Leprosy
  Leprosy11.jpeg: Leprosy
  Leprosy13.jpeg: Yaws
  Leprosy14.jpeg: Yaws
  Leprosy15.jpeg: Leprosy
  Leprosy16.jpeg: Mycetoma
  Leprosy17.jpeg: Yaws
  Leprosy18.jpeg: Leprosy
  Leprosy19.jpeg: Leprosy
  Leprosy2.jpg: Lymphatic Filariasis
  Leprosy20.jpeg: Mycetoma
  Leprosy4.jpg: Buruli Ulcers
  Leprosy5.jpg: Scabis
  Leprosy6.jpg: Buruli Ulcers
  Leprosy7.jpeg: Scabis
  Lymphatic Filariasis10.jpeg: Leprosy
  Lymphatic Filariasis6.jpg: Lympha